Import des données

In [ ]:
import pandas as pd
df = pd.read_csv(
'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

df.head(2)

: 

Imports utiles pour le DM

In [ ]:
!pip install -r requirements.txt

In [ ]:
import great_tables as gt
from great_tables import GT, style, loc


### Question 1 


On doit mettre à jour la variable code_commune en utilisant la variable actuelle et la variable code_departement. Pour cela, on va devoir concaténer les deux variables de telle sorte à obtenir ensuite un code à 5 chiffres. Les deux premiers chiffres correspondront au département (par exemple : 01 pour l'Ain, 59 pour le Nord) et les trois derniers au code commune (par exemple : 049 pour le code commune 49 dans le cas de Montrouge).

In [ ]:
df['code_commune'] = df['code_commune'].astype(str).str.zfill(3)

df['code_commune'] = df['code_departement'].astype(str) + df['code_commune']

df.head(20)

Nous devons également créer une colonne candidat qui comporte les prénoms et les noms de tous les candidats. Pour cela, nous allons concaténer les colonnes prenom et nom.

In [ ]:
df["candidat"] = df["prenom"].astype(str) + " " + df["nom"].astype(str)

print(df['candidat'].unique())

### Question 2

On souhaite déterminer le nombre de candidats qu'il y avait aux élections présidentielles de 2022. Pour cela, on va compter le nombre de candidats différents en excluant les votes blancs, les votes nuls et l'abstention.

In [ ]:
candidats = len(df[~df.isin(['nan abstentions', 'nan blancs', 'nan nuls'])]['candidat'].unique())

f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."

### Question 3


In [ ]:
exclus = ['nan abstentions', 'nan blancs', 'nan nuls']
df_exprimes = df[~df['candidat'].isin(exclus)]

scores = (
    df_exprimes
    .groupby('candidat', as_index=False)['voix']
    .sum()
    .sort_values('voix', ascending=False)
)

total_exprimes = scores['voix'].sum()
scores['pourcentage'] = (scores['voix'] / total_exprimes * 100).round(2)

(
    GT(scores)
    .tab_header(title="Résultats du 1er tour - Présidentielle 2022")
    .cols_label(candidat="Candidat", voix="Voix", pourcentage="% votes exprimés")
    .fmt_number(columns="voix", sep_mark=" ", dec_mark=",", decimals=0)
    .fmt_number(columns="pourcentage", decimals=2, dec_mark=",")
    .cols_align(align="right", columns=["voix", "pourcentage"])
)

# 2. Comparaison des scores départements aux moyennes nationales.

### Question 4 

In [ ]:
total_dep = df_exprimes.groupby('code_departement')['voix'].sum().rename('total_dep')


score_departements = (
    df_exprimes
    .groupby(['code_departement', 'libelle_departement', 'candidat'], as_index=False)['voix']
    .sum()
)

score_departements = score_departements.merge(total_dep, on='code_departement')
score_departements['pourcentage'] = (score_departements['voix'] / score_departements['total_dep'] * 100).round(2)
score_departements = score_departements.drop(columns='total_dep')

score_departements.head(10).sort_values('pourcentage', ascending=False)

In [ ]:
score_departements[score_departements['code_departement'] == '11'].sort_values('pourcentage', ascending=False)

### Question 5

In [ ]:
score_departements = score_departements.rename(columns={'voix': 'voix_departement', 'pourcentage': 'score_departement'})
scores = scores.rename(columns={'voix' : 'voix_national', 'pourcentage':'score_national'})
question5 = pd.merge(
    score_departements, scores, left_on="candidat", right_on="candidat", how="left", sort=False
)


question5.head(10)

### Question 6 

In [ ]:
question5['surrepresentation'] = (
    (question5['score_departement'] - question5['score_national']) / question5['score_national'] * 100
).round(2)

question5.head()

### Question 7

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def plot_surrepresentation(nom_candidat, top_n=5):
    df_cand = question5[question5['candidat'].str.contains(nom_candidat)]
    data = df_cand.reindex(df_cand['surrepresentation'].abs().nlargest(top_n).index)
    data = data.sort_values('surrepresentation')
   
    plt.figure()
    plt.barh(data['code_departement'], data['surrepresentation'])
    plt.axvline(0, color='steelblue', linewidth=0.8)
    plt.xlabel('Surreprésentation')
    plt.ylabel('Département')
    plt.title(f'Top {top_n} des surreprésentations de {nom_candidat}')
    plt.tight_layout()
    plt.show()

plot_surrepresentation('POUTOU')

# 3. Un peu de cartographie

### Question 8

In [ ]:
from cartiflette import carti_download
departement_borders = carti_download(
values = ["France"],
crs = 4326,
borders ="DEPARTEMENT",
vectorfile_format="geojson",
simplification=50,
filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
source="EXPRESS-COG-CARTO-TERRITOIRE",
year=2022)

In [ ]:
from visualisation import carte_candidat

In [ ]:
carte_candidat(question5, departement_borders, 'LE PEN');